# Retrieval, holding the negatives fixed

One heatmap per block, side by side. Rows are the generator that wrote the *negative*
ideas; columns are judge models; a cell is what adding related-work retrieval to the
judge prompt did to that judge on negatives from that generator — the change on top,
and the score it landed on in parentheses underneath.

Every row is a pair of runs that differ only in retrieval, so a cell is the retrieval
effect with the generator held fixed and nothing else. Rows run weakest negatives at the
top to strongest at the bottom, so the column reads as a difficulty gradient.

The rows come from different merges: `sonnet-4-5` is the `retrieval` panel of
`merge_vanilla_all_metrics`, the rest are the `retrieval_given_<backbone>_<setup>`
merges. A different merge is a different test set, so the `sonnet-4-5` row is set apart
above the grid — the same treatment [negatives_source_figure.ipynb](negatives_source_figure.ipynb)
gives its human row, and the same thing only the caption can say out loud.

Numbers are read straight from the `unified_<setup>_<variant>.json` files a merge writes.
Nothing is recomputed here — no merge re-run, no bootstrap; `*` is the significance the
merge already stored.

Everything the figure says — rows, judges, metrics, titles, axis names, geometry — lives
in the one config cell below. It is drawn in the style of
[two_track_figure.ipynb](two_track_figure.ipynb): same palette, same typography, the same
inch-exact way of laying a cell out, and the same two-line cell.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "output").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

# The red→green palette, the typeface, the Type-42 font settings, the shade at
# which cell text turns white, the white left round a trimmed edge, where a merge
# keeps its chart data and the save path — all shared with every other figure, so
# a regenerated panel cannot drift from the ones beside it.
from novelty_eval.figures.common import (
    EDGE_IN, FONT_STACK, PDF_RCPARAMS, WHITE_TEXT_BAND, chart_data_path,
    delta_cmap, save_figure)

In [ ]:
# ---- everything the chart says, in one place ---------------------------------

# Which accuracy report the numbers came from: "filtered" or "unfiltered".
VARIANT = "filtered"

# The blocks of the figure, left to right: (block title, setup, metric key).
# The title is free text, and the only place the figure says which metric a block
# shows. Everything before the first newline is the block's name; the rest is the
# metric, set under it a step smaller and lighter — which number is being shown
# rather than what the block is.
# Metric keys (the next cell prints what each merge holds):
#   pointwise  accuracy, f1_macro, f1_pos, f1_neg,
#              precision_pos, precision_neg, recall_pos, recall_neg
#   pairwise   pairwise_accuracy, pairwise_accuracy_strict, pairwise_accuracy_no_ties
BLOCKS = [
    ("Pointwise\nMacro-F1", "pointwise", "f1_macro"),
    ("Pairwise\nStrict Accuracy", "pairwise", "pairwise_accuracy_strict"),
]

# Rows, top to bottom: which generator wrote the negative ideas, weakest first.
# (label, merge dir under output/ablation_sweeps, panel in it), with {setup}
# expanding per block. Each panel is one retrieval run measured against the same
# run without retrieval, so the two differ in nothing else.
ROWS = [
    ("sonnet-4-5", "merge_vanilla_all_metrics", "retrieval"),
    ("opus-4-5", "retrieval_given_opus-4-5_{setup}",
     "opus-4-5_backbone_retrieval_{setup}_vanilla"),
    ("gpt-5.1", "retrieval_given_gpt_5.1_{setup}",
     "gpt_5.1_backbone_retrieval_{setup}_vanilla"),
    ("gpt-5.4", "retrieval_given_gpt_5.4_{setup}",
     "gpt_5.4_backbone_retrieval_{setup}_vanilla"),
]

# Judge columns, left to right: name in the data -> column header. These are the
# two judges the backbone merges were run for.
JUDGES = {"claude-opus-4-6": "opus-4-6", "gpt-5.4": "gpt-5.4"}

# The row to set apart, drawn above the grid with a gap under it and labelled in
# italic: it is the reference generator, and it comes from a different merge, and
# so a different test set, than the rows below. None makes it an ordinary row.
REFERENCE_ROW = "sonnet-4-5"

# The same model names appear as generators in the rows and as judges in the
# columns, so both axes need naming. "" drops either title.
ROW_AXIS_LABEL = "Negatives source"
COL_AXIS_LABEL = "Judge model"

# The two keys, sharing the bottom line of the figure: what a cell holds, on the
# left, and what a mark on it means, on the right. Keep both short — they sit on
# one line, so between them they have about the width of the figure. "" drops
# either one.
CELL_NOTE = "Δ (without→with retrieval)"
SIG_NOTE = "* p < 0.05 (two-sided paired bootstrap)"

# Layout in inches, so a cell is the size it says it is: every gap on the page is
# one number here, and there is no layout engine to reverse-engineer when it looks
# wrong. Sized to sit in one column of a two-column paper (~3.2in) rather than
# across the text width: the type is a step smaller than two_track_figure's, and
# a column is only as wide as the widest thing printed in it.
COL_IN = 0.56                     # width of one judge column: the "(before→after)"
                                  # line is 0.39in at this size, the rest is air
CELL_IN = 0.33                    # height of one row: two lines of cell text,
                                  # the change over the scores it moved between
GAP_ROWS = 0.10                   # blank space under the reference row, in rows
GUTTER_IN = 0.14                  # between blocks
SUB_IN = 0.085                    # the metric line, above the grid
TITLE_IN = 0.17                   # the block name, above that
MARGIN_IN = {"left": 0.66,        # room for the row names and the axis title
             "right": 0.04,
             "top": 0.34,         # the two title lines
             "bottom": 0.46,      # the judge names, the column title, and the keys
             "xlabel": 0.22,      # the column title, up from the bottom edge
             "note": 0.06}        # the keys, below it
FONT_SIZES = {"delta": 6.4, "score": 4.8, "row": 6.2, "judge": 5.4,
              "title": 7.2, "subtitle": 4.6, "axis": 6.6, "note": 5.4}

# Output subdirectory under output/figures/, and the filename prefix — named like
# the sibling figures' directories (controlled-study, negatives-source). The
# variant and the metric keys land in the filename too, so changing what BLOCKS
# shows writes a new file instead of overwriting the last one.
FIGURE = "retrieval-given-backbone"
FORMATS = [".pdf", ".png"]        # PDF is vector, PNG is what displays inline

In [ ]:
PANELS = {
    (label, setup): json.loads(
        chart_data_path(ROOT / "output/ablation_sweeps" / merge.format(setup=setup),
                        setup, VARIANT).read_text()
    )["panels"][panel.format(setup=setup)]
    for label, merge, panel in ROWS
    for _, setup, _ in BLOCKS
}

# What each row is actually comparing. A mis-spelled panel is a KeyError above;
# a well-spelled one pointed at the wrong comparison is only visible here.
for (label, setup), panel in PANELS.items():
    print(f"{label:>10}  {setup:<9}  {panel['title']:<32}  vs {panel['baseline_name']}")

In [ ]:
def record(label, setup, judge, metric):
    """One cell: both scores, their difference, and whether it is significant."""
    panel = PANELS[(label, setup)]

    def score(side):
        """One side of the comparison, as a percentage. NaN when it has no number."""
        value = panel[side].get(judge, {}).get(metric)
        return np.nan if value is None else value * 100

    without, with_ = score("baseline_metrics"), score("ablation_metrics")
    # Bootstrap results are keyed "<judge>||<metric>" on disk.
    sig = panel["bootstrap_results"].get(f"{judge}||{metric}", {}).get("significant")
    return {"negatives": label, "judge": JUDGES[judge], "without": without,
            "with": with_, "delta": with_ - without, "significant": bool(sig)}


TABLES = [
    (title, pd.DataFrame([record(label, setup, judge, metric)
                          for label, _, _ in ROWS for judge in JUDGES]))
    for title, setup, metric in BLOCKS
]

# The whole data path, before any drawing: if a cell in the figure looks wrong,
# the reason is visible here. `without` is the only number the figure does not
# print, so it is the one to check a surprising delta against.
for title, table in TABLES:
    print(title)
    print(table.round(1).to_string(index=False), "\n")

In [ ]:
import matplotlib.pyplot as plt

# The shared stack, given as the family list rather than only as the sans-serif
# alias, so a glyph the first font lacks falls back to the next font that has
# it. Helvetica Neue has no →, and without the fallback every cell would print a
# missing-glyph box in its place. Nothing else about the typeface changes.
plt.rcParams.update({"font.family": FONT_STACK, "font.sans-serif": FONT_STACK,
                     **PDF_RCPARAMS})

# One scale for every number on the page, symmetric about zero, so a shade means
# the same change wherever it appears and the two blocks can be read side by side.
deltas = np.concatenate([t["delta"].to_numpy(float) for _, t in TABLES])
span = max(1.0, np.nanmax(np.abs(deltas)))
vmin, vmax = -span, span
mesh_kw = dict(cmap=delta_cmap(), vmin=vmin, vmax=vmax,
               edgecolors="white", linewidth=1.2)

def text_colour(v):
    """Cell text: white only on the dark ends of the scale."""
    shade = (v - vmin) / (vmax - vmin)
    dark = shade < WHITE_TEXT_BAND[0] or shade > WHITE_TEXT_BAND[1]
    return "white" if dark else "#1a1a1a"

# The reference row sits above the grid with a gap under it, so it never reads as
# one more backbone. Row i is centred on y = i (+ the gap, below it).
rows = [label for label, _, _ in ROWS]
n_ref = 1 if REFERENCE_ROW in rows else 0
rows = ([REFERENCE_ROW] if n_ref else []) + [r for r in rows if r != REFERENCE_ROW]
gap = GAP_ROWS if n_ref else 0.0
y = [i + 0.5 + (gap if i >= n_ref else 0.0) for i in range(len(rows))]
judges = list(JUDGES.values())
n_col, n_grid = len(judges), len(rows) - n_ref

grid_in = CELL_IN * (len(rows) + gap)
fig_h = grid_in + MARGIN_IN["top"] + MARGIN_IN["bottom"]
block_in = COL_IN * n_col
blocks_in = block_in * len(TABLES) + GUTTER_IN * (len(TABLES) - 1)
width_in = MARGIN_IN["left"] + blocks_in + MARGIN_IN["right"]

fig = plt.figure(figsize=(width_in, fig_h))
gs = fig.add_gridspec(1, len(TABLES), wspace=GUTTER_IN / block_in,
                      left=MARGIN_IN["left"] / width_in,
                      right=1 - MARGIN_IN["right"] / width_in,
                      top=1 - MARGIN_IN["top"] / fig_h,
                      bottom=MARGIN_IN["bottom"] / fig_h)

for k, (title, table) in enumerate(TABLES):
    ax = fig.add_subplot(gs[0, k])
    block = table.set_index(["negatives", "judge"])
    grid = lambda col: block[col].unstack().reindex(index=rows, columns=judges)
    delta, before, after = grid("delta"), grid("without"), grid("with")
    sig = grid("significant")

    # The colour is the change; the two scores it moved between are printed,
    # never coloured.
    cells = np.ma.masked_invalid(delta.to_numpy(float))
    x = np.arange(n_col + 1)
    if n_ref:
        ax.pcolormesh(x, np.arange(n_ref + 1), cells[:n_ref], **mesh_kw)
    ax.pcolormesh(x, n_ref + gap + np.arange(n_grid + 1), cells[n_ref:], **mesh_kw)

    for i, row in enumerate(rows):
        for j, judge in enumerate(judges):
            d = delta.loc[row, judge]
            if not np.isfinite(float(d)):
                ax.text(j + 0.5, y[i], "—", ha="center", va="center",
                        fontsize=6, color="#999999")
                continue
            # Weight is significance and nothing else, so the significant cells
            # are findable without reading a single asterisk.
            marked = bool(sig.loc[row, judge])
            colour = text_colour(d)
            ax.text(j + 0.5, y[i] - 0.16, f"{d:+.1f}{'*' if marked else ''}",
                    ha="center", va="center", fontsize=FONT_SIZES["delta"],
                    fontweight="bold" if marked else "normal", color=colour)
            # Where the change started and where it landed: the same +5 means
            # something different at 50 than at 85, and the without-retrieval
            # score is what says which. Small and italic, so the pair never
            # reads as a second delta; CELL_NOTE below says what it is.
            ax.text(j + 0.5, y[i] + 0.17,
                    f"({before.loc[row, judge]:.1f}→{after.loc[row, judge]:.1f})",
                    ha="center", va="center", fontsize=FONT_SIZES["score"],
                    style="italic", color=colour, alpha=0.72)

    ax.set_xlim(0, n_col)
    ax.set_ylim(len(rows) + gap, 0)                  # first row at the top
    # Set horizontally: the longest judge name is narrower than a cell at this
    # size, so there is nothing for a rotation to rescue.
    ax.set_xticks(x[:-1] + 0.5, judges, fontsize=FONT_SIZES["judge"])
    ax.set_yticks(y, rows, fontsize=FONT_SIZES["row"])
    # A block is named on two lines: the setup in the title's weight, and the
    # metric under it a step smaller and lighter, since it is which number is
    # being shown rather than what the block is. Plain text either way, as in
    # two_track_figure: the grid is the only thing that needs a shade of its own.
    head, _, metric_label = title.partition("\n")
    ax.set_title(head, fontsize=FONT_SIZES["title"], fontweight="bold",
                 pad=TITLE_IN * 72)
    if metric_label:
        ax.text(0.5, 1 + SUB_IN / grid_in, metric_label.replace("\n", " "),
                transform=ax.transAxes, ha="center", va="center",
                fontsize=FONT_SIZES["subtitle"], color="#555555")
    ax.tick_params(length=0, pad=2)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # The row names and both axis titles belong to the left-hand block only.
    if k:
        ax.tick_params(labelleft=False)
        continue
    for label in ax.get_yticklabels()[:n_ref]:
        label.set(style="italic", color="#555555")   # a different test set
    if ROW_AXIS_LABEL:
        ax.set_ylabel(ROW_AXIS_LABEL, fontsize=FONT_SIZES["axis"],
                      fontweight="bold", color="#333333", labelpad=5)

# A block title can be wider than a two-judge block. Widen the figure by what the
# rightmost one overruns, so the key below it still sets flush with the widest
# ink rather than under the middle of the title.
fig.canvas.draw()
ink_in = max(ax.get_tightbbox(fig.canvas.get_renderer()).x1
             for ax in fig.axes) / fig.dpi
width_in = max(width_in, ink_in + MARGIN_IN["right"])
fig.set_size_inches(width_in, fig_h)
gs.update(left=MARGIN_IN["left"] / width_in,
          right=(MARGIN_IN["left"] + blocks_in) / width_in)

if COL_AXIS_LABEL:
    fig.text((MARGIN_IN["left"] + blocks_in / 2) / width_in,
             MARGIN_IN["xlabel"] / fig_h, COL_AXIS_LABEL, ha="center",
             va="bottom", fontsize=FONT_SIZES["axis"], fontweight="bold",
             color="#333333")
# The two keys share the bottom line — what a cell holds on the left, what a mark
# means on the right — so neither costs the figure a line of its own. They are
# set as dark as the axis titles and as large as the judge names, so the key to
# the asterisk is read rather than skipped. Keep both short enough not to meet
# in the middle.
if CELL_NOTE:
    fig.text(MARGIN_IN["left"] / width_in, MARGIN_IN["note"] / fig_h, CELL_NOTE,
             ha="left", va="bottom", fontsize=FONT_SIZES["note"], color="#333333")
if SIG_NOTE:
    fig.text(1 - MARGIN_IN["right"] / width_in, MARGIN_IN["note"] / fig_h,
             SIG_NOTE, ha="right", va="bottom", fontsize=FONT_SIZES["note"],
             color="#333333")

# The margins above are what holds the type apart *inside* the figure; whatever
# white is left over at the edges is not. Saved trimmed to the ink with EDGE_IN
# round it, the same edge the sibling figures leave, so the page around the
# figure is the document's to set and not this notebook's.
stem = "_".join([FIGURE, VARIANT] + [metric for _, _, metric in BLOCKS])
out_path = ROOT / "output/figures" / FIGURE / (stem + FORMATS[0])
written = save_figure(fig, out_path, formats=FORMATS[1:],
                      bbox_inches="tight", pad_inches=EDGE_IN)
print(*[p.relative_to(ROOT) for p in written], sep="\n")
display(Image(filename=str(next(p for p in written if p.suffix == ".png"))))